In [1]:
%pip install -q langchain langchain_openai langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [2]:
import os
try:
    # In Colab? read from userdata (secrets)
    from google.colab import userdata
    ON_COLAB = True
    os.environ["SDAIA"] = userdata.get("SDAIA")
except ImportError:
    # Load `.env` file (locally)
    from dotenv import load_dotenv
    load_dotenv(override=True)

In [3]:
from langchain_openai import ChatOpenAI

In [5]:

model_nemotron3_nano_precise = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0,
    # OpenRouter instead of the default OpenAI API
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("SDAIA"),
)



In [ ]:
from pydantic import BaseModel, Field
from typing import Optional
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

#Schema
class Experience(BaseModel):
    company:     str
    role:        str
    duration:    Optional[str] = None
    description: Optional[str] = None

class ResumeData(BaseModel):
    candidate_name: str               = Field(description="Full name of the candidate")
    email:          Optional[str]     = Field(description="Email address")
    phone:          Optional[str]     = Field(description="Phone number")
    skills:         list[str]         = Field(description="Technical and soft skills")
    experience:     list[Experience]  = Field(description="Work experience entries")
    education:      Optional[list[str]] = Field(description="Degrees or certifications")
    languages:      Optional[list[str]] = Field(description="Spoken languages")
    summary:        Optional[str]     = Field(description="Professional summary or objective")

#Load Resume
with open("resume.txt", "r", encoding="utf-8") as f:
    resume_text = f.read()

# ── Build Messages ────────────────────────────────────────────────────────────
messages = [
    SystemMessage(
        content=(
            "You are an expert HR assistant. "
            "Extract structured data from the resume provided by the user. "
            "If a field is not found, return null."
        )
    ),
    HumanMessage(
        content=resume_text
    ),
]

# ── Invoke ────────────────────────────────────────────────────────────────────
structured_model = model_nemotron3_nano_precise.with_structured_output(ResumeData)
result: ResumeData = structured_model.invoke(messages)

# Inspect the raw AIMessage metadata (before structured parsing)
raw_response: AIMessage = model_nemotron3_nano_precise.invoke(messages)
print("=== AIMessage Metadata ===")
print(f"  ID             : {raw_response.id}")
print(f"  Role           : {raw_response.type}")
print(f"  Response Meta  : {raw_response.response_metadata}")
print(f"  Usage Metadata : {raw_response.usage_metadata}\n")

# ── Display Structured Result ─────────────────────────────────────────────────
print("=== Parsed Resume ===")
print(f"👤 Name      : {result.candidate_name}")
print(f"📧 Email     : {result.email}")
print(f"📞 Phone     : {result.phone}")
print(f"📝 Summary   : {result.summary}\n")

print("🛠️  Skills:")
for skill in result.skills:
    print(f"   • {skill}")

print("\n💼 Experience:")
for exp in result.experience:
    print(f"   [{exp.duration}] {exp.role} @ {exp.company}")
    if exp.description:
        print(f"      ↳ {exp.description}")

print("\n🎓 Education:")
for edu in result.education or []:
    print(f"   • {edu}")

print("\n🌐 Languages:")
for lang in result.languages or []:
    print(f"   • {lang}")